# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets and fields (by `@id`), as described in the metadata.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}: @id = {rs.id_}, name = {rs.name}")
    for field in rs.fields:
        print(f"    Field: @id = {field.id_}, name = {field.name}, type = {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We will use the record set and fields' `@id`s found above. For this demonstration, we'll use the primary record set (typically containing the main table of records), which in this dataset is likely named something like `cr:RecordSet/0`/`cr:RecordSet_1` or similar—let's inspect the first one and use its `@id`.

In [ ]:
# Extract data from the first (typically main) record set
if len(record_sets) == 0:
    raise RuntimeError("No record sets found in this dataset.")
first_rs = record_sets[0]
first_rs_id = first_rs.id_  # This is the @id string
print(f"Using record set: {first_rs_id}\n")

# If there are multiple record sets, collect their @id's
record_set_ids = [rs.id_ for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"Columns available in {first_rs_id}:")
print(dataframes[first_rs_id].columns.tolist())
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering, normalization, and grouping.

_Note: All field names are referenced by their `@id` as specified in the Croissant schema. Make sure to replace the placeholder below with the actual `@id` of the numeric field to analyze. For this dataset, let's pick a common protein attribute such as `cr:field/Abundance` if it exists._

In [ ]:
# Example field choice: use the actual @id from the data overview (e.g., 'cr:field/Abundance')
numeric_field_id = None

# Try to auto-detect likely numeric columns
likely_numeric_columns = [c for c in dataframes[first_rs_id].columns if c.lower() in ["abundance", "mw", "coverage", "peptide_count", "cr:field/abundance", "cr:field/mw", "cr:field/coverage", "cr:field/peptide_count"]]
if likely_numeric_columns:
    numeric_field_id = likely_numeric_columns[0]
else:
    # Fallback to first numeric-looking column
    for col in dataframes[first_rs_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[first_rs_id][col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}")

df = dataframes[first_rs_id].copy()

# Filter (e.g., keep proteins with abundance > 10)
threshold = 10
if numeric_field_id is not None:
    # Try converting to numeric, coercing errors
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors="coerce")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: (total = {len(filtered_df)})")
    print(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try grouping by a categorical column (e.g., a sample or modification field)
    group_field_candidates = [c for c in df.columns if c.lower() in ["sample", "modification", "group", "cr:field/sample", "cr:field/modification"]]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No obvious group field found for grouping.")
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show a histogram of the numeric field (if present), and a boxplot grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    filtered_df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if available
    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load FAIR² dataset metadata and records using Croissant and `mlcroissant`
- Inspect record sets and field structures using `@id`
- Extract records into Pandas DataFrames using record set `@id`s
- Filter, normalize, and group data dynamically based on field types
- Visualize attribute distributions and group differences

This approach enables reproducible, schema-driven dataset access and processing, which is especially useful for multi-source or FAIR-compliant scientific record sets. For deeper analysis, customize the EDA and visualization steps for your particular research hypotheses or use cases.